# 00 · Common module (config, dataset, model, losses, trainer)
Other notebooks load this with `%run ./00_common.ipynb`. You do not need to run it by hand.

**Model summary** — `PENet` implements the proposed pipeline:
`CTPA volume → vessel mask → vessel graph + candidate ROIs → multi-scale 3D encoder → cross-scale fusion → vessel-topology attention → multi-task heads (detect / localise / burden+severity / vessel) → false-positive suppression → MC-dropout uncertainty → explainable output`.

Supervision available in RSNA-STR is *weak* (study labels + slice-level PE flags, no voxel masks, no vessel masks, no clot volumes). Therefore:
* **Localisation** is trained with slice-level multiple-instance learning; in-plane localisation is a weakly supervised heat-map restricted to the vessel tree.
* **Burden** is a research-derived index; the training proxy is slice-extent (fraction of vessel-bearing slices that are PE-positive).
* **Vessel segmentation** is a classical pseudo-label, used as an auxiliary target.

In [ ]:
import os, sys, json, math, random, glob, time, warnings
from dataclasses import dataclass, field, asdict
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")

IS_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if IS_KAGGLE else "./work"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NF = 14   # node-feature dimension produced by 02_vessel_topology.ipynb
LABELS = ["pe_present", "rv_lv_ge1", "leftsided_pe", "rightsided_pe", "central_pe", "chronic_pe"]

@dataclass
class Cfg:
    # ---- paths
    tfrec_glob: str = "/kaggle/input/rsna-str-tfrec/**/*.tfrec"
    labels_csv: str = ""                      # optional: official RSNA train.csv (image- and study-level labels)
    work: str = WORK
    # ---- data
    vol_shape: tuple = (64, 128, 128)         # (D,H,W) after resampling. Increase if GPU memory allows.
    n_roi: int = 32                           # max candidate embolus ROIs per study
    max_hop: int = 3                          # graph-distance buckets for topology attention
    cand_thr: float = 0.0                     # candidate-selection threshold on vessel-dip score (tunable)
    # ---- model
    embed: int = 96
    heads: int = 4
    n_layers: int = 2
    dropout: float = 0.2
    ch: tuple = (16, 32, 64, 128)
    gate_sigma: float = 0.12                  # FP-suppression radius (normalised coords)
    # ---- optimisation
    lr: float = 3e-4
    wd: float = 1e-2
    bs: int = 2
    accum: int = 4
    epochs: int = 20
    patience: int = 6
    pos_weight: float = 1.0
    slice_pos_weight: float = 3.0
    num_workers: int = 2
    seed: int = 42
    # ---- augmentation (tunable)
    aug_noise: float = 0.02
    aug_contrast: float = 0.10
    aug_bright: float = 0.05
    # ---- loss weights (tunable)
    w: dict = field(default_factory=lambda: dict(det=1.0, loc=1.0, burden=0.5, sev=0.5, vessel=0.3, tok=0.5, fp=0.5))
    # ---- decision thresholds (tunable)
    thr_det: float = 0.5
    thr_slice: float = 0.5
    burden_bins: tuple = (0.10, 0.25)         # low < 0.10 <= moderate < 0.25 <= high  (research-derived, NOT clinically validated)

    @property
    def vol_dir(self): return f"{self.work}/cache/vol"
    @property
    def vessel_dir(self): return f"{self.work}/cache/vessel"
    @property
    def ckpt_dir(self): return f"{self.work}/ckpt"
    @property
    def out_dir(self): return f"{self.work}/results"

@dataclass
class Flags:
    """Switches used by the ablation study (Experiment B). Full model = all True."""
    use_vessel: bool = True     # vessel mask channel + vessel-graph candidate tokens + node features
    multiscale: bool = True     # sample fine/mid/coarse feature maps and fuse them
    topo_attn: bool = True      # graph-distance bias in token attention
    multitask: bool = True      # localisation / burden / severity / vessel / token heads
    fp_suppress: bool = True    # second-stage false-positive suppression gate
    @property
    def name(self):
        return "+".join(k for k, v in asdict(self).items() if v) or "baseline"

cfg = Cfg()
for d in (cfg.vol_dir, cfg.vessel_dir, cfg.ckpt_dir, cfg.out_dir): os.makedirs(d, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(cfg.seed)
print("device:", DEVICE, "| work dir:", cfg.work)

In [ ]:
# ============================== DATASET ==============================
def load_splits():
    return json.load(open(f"{cfg.work}/splits.json"))

class PEDataset(Dataset):
    """Reads cached volume (01) + vessel graph candidates (02)."""
    def __init__(self, ids, cfg, train=False, perturb=None):
        self.ids, self.cfg, self.train, self.perturb = list(ids), cfg, train, perturb
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        c = self.cfg; sid = self.ids[i]
        a = np.load(f"{c.vol_dir}/{sid}.npz"); b = np.load(f"{c.vessel_dir}/{sid}.npz")
        vol = torch.from_numpy(a["vol"].astype(np.float32))[None]                  # (1,D,H,W) in [0,1]
        ves = torch.from_numpy(b["vessel"].astype(np.float32))[None]
        if self.perturb is not None: vol = self.perturb(vol)                        # robustness tests (06)
        if self.train:
            vol = vol * (1 + c.aug_contrast * (2 * torch.rand(1) - 1)) + c.aug_bright * (2 * torch.rand(1) - 1)
            vol = vol + c.aug_noise * torch.randn_like(vol)
            vol = vol.clamp(0, 1)
        y = a["y"].astype(np.float32); ym = ~np.isnan(y); y = np.nan_to_num(y)
        sl = a["slice_lab"].astype(np.float32); sv = float(not np.isnan(sl).any()); sl = np.nan_to_num(sl)
        K = c.n_roi
        valid = b["roi_valid"][:K].copy() & (b["roi_score"][:K] >= c.cand_thr)
        if not valid.any(): valid[0] = True                                         # always keep >=1 token
        has = (b["vessel"].sum((1, 2)) > 0).astype(np.float32)
        burden_t = float((sl * has).sum() / max(has.sum(), 1.0))                    # slice-extent burden proxy
        return dict(sid=sid, vol=vol, vessel=ves, y=torch.from_numpy(y), y_mask=torch.from_numpy(ym.astype(np.float32)),
                    slice_lab=torch.from_numpy(sl), slice_valid=torch.tensor(sv), burden_t=torch.tensor(burden_t),
                    centers=torch.from_numpy(b["roi_centers"][:K]), nfeat=torch.from_numpy(b["roi_feats"][:K]),
                    hops=torch.from_numpy(b["roi_hops"][:K, :K].astype(np.int64)), valid=torch.from_numpy(valid))

def make_loader(ids, cfg, train, bs=None, shuffle=None, perturb=None):
    return DataLoader(PEDataset(ids, cfg, train, perturb), batch_size=bs or (cfg.bs if train else max(cfg.bs, 2)),
                      shuffle=train if shuffle is None else shuffle, num_workers=cfg.num_workers,
                      pin_memory=DEVICE.type == "cuda", drop_last=train)

def to_dev(b):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}

In [ ]:
# ============================== MODEL ==============================
def conv_block(i, o, s=1):
    return nn.Sequential(nn.Conv3d(i, o, 3, s, 1, bias=False), nn.InstanceNorm3d(o, affine=True), nn.LeakyReLU(0.1, True),
                         nn.Conv3d(o, o, 3, 1, 1, bias=False), nn.InstanceNorm3d(o, affine=True), nn.LeakyReLU(0.1, True))

class TopoBlock(nn.Module):
    """Transformer block whose attention logits receive a learnable bias indexed by vessel-graph hop distance
    -> tokens on connected/nearby vessel segments attend to each other preferentially (Vessel-Topology Attention)."""
    def __init__(s, C, heads, max_hop, dropout, topo):
        super().__init__()
        s.h, s.d, s.topo = heads, C // heads, topo
        s.n1, s.n2 = nn.LayerNorm(C), nn.LayerNorm(C)
        s.qkv, s.proj = nn.Linear(C, 3 * C), nn.Linear(C, C)
        s.hop_bias = nn.Embedding(max_hop + 2, heads) if topo else None
        s.mlp = nn.Sequential(nn.Linear(C, 2 * C), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * C, C))
        s.drop = nn.Dropout(dropout)
    def forward(s, x, hops, valid):
        B, K, C = x.shape
        q, k, v = s.qkv(s.n1(x)).view(B, K, 3, s.h, s.d).permute(2, 0, 3, 1, 4)        # (B,h,K,d)
        lg = (q @ k.transpose(-1, -2)).float() / math.sqrt(s.d)
        if s.topo: lg = lg + s.hop_bias(hops).permute(0, 3, 1, 2).float()
        lg = lg.masked_fill(~valid[:, None, None, :], -1e4)
        a = s.drop(torch.softmax(lg, -1))
        o = (a.to(v.dtype) @ v).transpose(1, 2).reshape(B, K, C)
        x = x + s.drop(s.proj(o))
        x = x + s.drop(s.mlp(s.n2(x)))
        return x, a

class Decoder(nn.Module):
    def __init__(s, ch, out=2):
        super().__init__()
        c1, c2, c3, c4 = ch
        s.u3, s.u2, s.u1 = conv_block(c4 + c3, c3), conv_block(c3 + c2, c2), conv_block(c2 + c1, c1)
        s.out = nn.Conv3d(c1, out, 1)
    def forward(s, f1, f2, f3, f4):
        up = lambda x, r: F.interpolate(x, size=r.shape[2:], mode="trilinear", align_corners=False)
        x = s.u3(torch.cat([up(f4, f3), f3], 1)); x = s.u2(torch.cat([up(x, f2), f2], 1)); x = s.u1(torch.cat([up(x, f1), f1], 1))
        return s.out(x)

def sample_at(f, centers):
    """Trilinear sampling of feature map f (B,C,D,H,W) at token centres (B,K,3 in xyz in [-1,1]) -> (B,K,C)."""
    g = centers[:, :, None, None, :].float()
    return F.grid_sample(f.float(), g, mode="bilinear", align_corners=True)[..., 0, 0].transpose(1, 2)

def gaussian_field(centers, weights, shape, sigma):
    """max_k weights_k * exp(-|x-c_k|^2 / 2 sigma^2) on a low-res grid (B,1,D,H,W). Used for the FP gate and attention maps."""
    B, K, _ = centers.shape; D, H, W = shape
    zs, ys, xs = [torch.linspace(-1, 1, n, device=centers.device) for n in (D, H, W)]
    gz, gy, gx = torch.meshgrid(zs, ys, xs, indexing="ij")
    grid = torch.stack([gx, gy, gz], -1).view(1, 1, -1, 3)
    d2 = ((grid - centers.float()[:, :, None, :]) ** 2).sum(-1)
    return (weights.float()[:, :, None] * torch.exp(-d2 / (2 * sigma ** 2))).max(1).values.view(B, 1, D, H, W)

class PENet(nn.Module):
    """Vessel-topology-aware multi-task 3D network (see architecture diagram in README)."""
    def __init__(s, cfg, fl=Flags()):
        super().__init__()
        s.cfg, s.fl = cfg, fl
        c1, c2, c3, c4 = cfg.ch; C = cfg.embed
        s.e1 = conv_block(2 if fl.use_vessel else 1, c1); s.e2 = conv_block(c1, c2, 2)
        s.e3 = conv_block(c2, c3, 2); s.e4 = conv_block(c3, c4, 2)
        s.gproj = nn.Sequential(nn.Linear(2 * c4, C), nn.GELU())
        zdim = C
        if fl.use_vessel:
            scales = [(c1, "p1"), (c2, "p2"), (c3, "p3")] if fl.multiscale else [(c3, "p3")]
            s.scale_names = [n for _, n in scales]
            for ci, n in scales: setattr(s, n, nn.Linear(ci, C))
            s.node_mlp = nn.Sequential(nn.Linear(NF, C), nn.GELU(), nn.Linear(C, C))
            s.pos_mlp = nn.Linear(3, C)
            s.blocks = nn.ModuleList([TopoBlock(C, cfg.heads, cfg.max_hop, cfg.dropout, fl.topo_attn) for _ in range(cfg.n_layers)])
            s.tok_norm = nn.LayerNorm(C); s.pool_score = nn.Linear(C, 1)
            zdim = 2 * C
        s.drop = nn.Dropout(cfg.dropout)
        s.det_head = nn.Sequential(nn.Linear(zdim, C), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(C, 1))
        if fl.multitask:
            s.dec = Decoder(cfg.ch, 2)                                   # ch0: embolus heat-map, ch1: vessel auxiliary
            s.sev_head = nn.Sequential(nn.Linear(zdim, C), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(C, 5))
            s.burden_head = nn.Sequential(nn.Linear(zdim, C // 2), nn.GELU(), nn.Linear(C // 2, 1))
            if fl.use_vessel:
                s.cand_head = nn.Linear(C, 1)
                if fl.fp_suppress:
                    s.fp_head = nn.Sequential(nn.Linear(C + 3, C // 2), nn.GELU(), nn.Linear(C // 2, 1))

    def forward(s, b, keep_feat=False):
        fl, cfg = s.fl, s.cfg
        x, ves = b["vol"], b["vessel"]
        B = x.shape[0]
        f1 = s.e1(torch.cat([x, ves], 1) if fl.use_vessel else x); f2 = s.e2(f1); f3 = s.e3(f2); f4 = s.e4(f3)
        g = s.gproj(torch.cat([f4.mean((2, 3, 4)), f4.amax((2, 3, 4))], 1))
        out = {}; tok = None; z = g
        if fl.use_vessel:
            centers, valid = b["centers"], b["valid"]
            maps = {"p1": f1, "p2": f2, "p3": f3}
            T = torch.stack([getattr(s, n)(sample_at(maps[n], centers)) for n in s.scale_names], 2)   # (B,K,S,C)
            if T.shape[2] > 1:                                                                         # cross-scale fusion
                w = torch.softmax((T * g[:, None, None, :]).sum(-1) / math.sqrt(T.shape[-1]), 2)
                t = (w.unsqueeze(-1) * T).sum(2)
            else: t = T[:, :, 0]
            t = t + s.node_mlp(b["nfeat"]) + s.pos_mlp(centers)
            hops = b["hops"].clamp(max=cfg.max_hop + 1)
            for blk in s.blocks: t, att = blk(t, hops, valid)
            tok = s.tok_norm(t)
            imp = att.mean(1).mean(1) * valid.float()                                                  # attention received per token
            out["tok_imp"] = imp / imp.sum(1, keepdim=True).clamp(min=1e-6)
            pw = torch.softmax(s.pool_score(tok).squeeze(-1).float().masked_fill(~valid, -1e4), 1)
            z = torch.cat([g, (pw.unsqueeze(-1).to(tok.dtype) * tok).sum(1)], 1)
        zz = s.drop(z)
        det = s.det_head(zz).squeeze(-1); out["det_logit"] = det
        if keep_feat: out["f4"] = f4
        if not fl.multitask: return out
        out["sev_logits"] = s.sev_head(zz)
        out["burden_head"] = torch.sigmoid(s.burden_head(zz).squeeze(-1))
        h = s.dec(f1, f2, f3, f4)
        heat_logit, out["vessel_logit"] = h[:, 0:1], h[:, 1:2]
        vd = F.max_pool3d(ves, 3, 1, 1) if fl.use_vessel else torch.ones_like(ves)                  # restrict to (dilated) vessels
        hl = torch.where(vd > 0.5, heat_logit, torch.full_like(heat_logit, -10.0))
        heat = torch.sigmoid(hl.float())
        out["heat_logit"] = hl
        out["slice_logit"] = hl.float().flatten(3).squeeze(1).topk(16, -1).values.mean(-1)          # (B,D) top-k MIL pooling
        if fl.use_vessel:
            out["cand_logit"] = s.cand_head(tok).squeeze(-1)
            if fl.fp_suppress:
                hs = sample_at(hl.detach(), b["centers"])
                ki = torch.cat([tok, out["cand_logit"].detach().unsqueeze(-1).to(tok.dtype), hs.to(tok.dtype),
                                det.detach()[:, None, None].expand(-1, tok.shape[1], 1).to(tok.dtype)], -1)
                out["keep_logit"] = s.fp_head(ki).squeeze(-1)
                low = tuple(max(n // 4, 2) for n in x.shape[2:])
                rej = (1 - torch.sigmoid(out["keep_logit"].float())) * b["valid"].float()
                gate = 1 - gaussian_field(b["centers"], rej, low, cfg.gate_sigma)
                heat = heat * F.interpolate(gate, size=x.shape[2:], mode="trilinear", align_corners=False)
        out["heat"] = heat
        out["slice_prob"] = heat.amax((3, 4)).squeeze(1)                                            # (B,D) final per-slice score
        has = (ves.sum((3, 4)).squeeze(1) > 0).float()
        out["burden_ext"] = (out["slice_prob"] * has).sum(1) / has.sum(1).clamp(min=1)               # slice-extent burden index
        out["burden_vol"] = (heat * ves).sum((1, 2, 3, 4)) / ves.sum((1, 2, 3, 4)).clamp(min=1)      # volumetric burden index
        return out

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6

In [ ]:
# ============================== LOSSES ==============================
def token_labels(centers, slice_lab):
    """Weak token label: token is positive if its z-slice (+-1) is labelled PE."""
    D = slice_lab.shape[1]
    sl = F.max_pool1d(slice_lab[:, None], 3, 1, 1)[:, 0]
    zi = ((centers[..., 2] + 1) / 2 * (D - 1)).round().long().clamp(0, D - 1)
    return torch.gather(sl, 1, zi)

def dice_loss(logit, tgt, eps=1.0):
    p = torch.sigmoid(logit.float()); inter = (p * tgt).sum((1, 2, 3, 4))
    return (1 - (2 * inter + eps) / (p.sum((1, 2, 3, 4)) + tgt.sum((1, 2, 3, 4)) + eps)).mean()

def masked_mean(l, m): return (l * m).sum() / m.sum().clamp(min=1)

def compute_loss(out, b, cfg):
    """L = w_det L_det + w_loc L_loc + w_burden L_burden + w_sev L_sev + w_vessel L_vessel + w_tok L_tok + w_fp L_fp.
    Missing heads (ablation) are simply skipped."""
    w, L = cfg.w, {}
    pw = torch.tensor([cfg.pos_weight], device=DEVICE)
    L["det"] = masked_mean(F.binary_cross_entropy_with_logits(out["det_logit"].float(), b["y"][:, 0], pos_weight=pw, reduction="none"), b["y_mask"][:, 0])
    if "sev_logits" in out:
        L["sev"] = masked_mean(F.binary_cross_entropy_with_logits(out["sev_logits"].float(), b["y"][:, 1:6], reduction="none"), b["y_mask"][:, 1:6])
    if "slice_logit" in out:
        spw = torch.tensor([cfg.slice_pos_weight], device=DEVICE)
        l = F.binary_cross_entropy_with_logits(out["slice_logit"], b["slice_lab"], pos_weight=spw, reduction="none").mean(1)
        L["loc"] = masked_mean(l, b["slice_valid"])
        L["vessel"] = F.binary_cross_entropy_with_logits(out["vessel_logit"].float(), b["vessel"]) + dice_loss(out["vessel_logit"], b["vessel"])
        L["burden"] = F.smooth_l1_loss(out["burden_ext"], b["burden_t"], beta=0.05) + F.smooth_l1_loss(out["burden_head"], b["burden_t"], beta=0.05)
    if "cand_logit" in out:
        tl = token_labels(b["centers"], b["slice_lab"]); vm = b["valid"].float() * b["slice_valid"][:, None]
        L["tok"] = masked_mean(F.binary_cross_entropy_with_logits(out["cand_logit"].float(), tl, reduction="none"), vm)
        if "keep_logit" in out:
            L["fp"] = masked_mean(F.binary_cross_entropy_with_logits(out["keep_logit"].float(), tl, reduction="none"), vm)
    total = sum(w[k] * v for k, v in L.items())
    return total, {k: float(v) for k, v in L.items()}

In [ ]:
# ============================== METRICS / TRAIN / PREDICT ==============================
from sklearn.metrics import roc_auc_score, average_precision_score

def safe_auc(y, p):
    y = np.asarray(y); return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else float("nan")

def runs(binary):
    """Contiguous runs of 1s -> list of (start, end) inclusive. Used as 'lesion' extents along z."""
    r, s = [], None
    for i, v in enumerate(np.r_[binary, 0]):
        if v and s is None: s = i
        if not v and s is not None: r.append((s, i - 1)); s = None
    return r

def slice_f1(prob, lab, valid, thr):
    m = valid > 0
    if m.sum() == 0: return float("nan")
    p, l = (prob[m] >= thr), lab[m] > 0.5
    tp, fp, fn = (p & l).sum(), (p & ~l).sum(), (~p & l).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))

def best_slice_thr(prob, lab, valid):
    grid = np.linspace(0.05, 0.95, 19); f = [slice_f1(prob, lab, valid, t) for t in grid]
    return float(grid[int(np.nanargmax(f))]), float(np.nanmax(f))

def fp_lesions_per_negative(prob, y_pe, thr):
    neg = y_pe < 0.5
    return float(np.mean([len(runs(p >= thr)) for p in prob[neg]])) if neg.any() else float("nan")

def enable_mc_dropout(model):
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.train()

@torch.no_grad()
def predict(model, loader, mc=0, keep_heat=False, fwd=None):
    """Returns dict of numpy arrays. mc>0 -> MC-dropout (mean + std of detection prob)."""
    fwd = fwd or (lambda m, b: m(b)); R = defaultdict(list)
    ac = torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda")
    for b in loader:
        b = to_dev(b)
        model.eval()
        with ac: out = fwd(model, b)
        R["sid"] += list(b["sid"]); R["det_prob"].append(torch.sigmoid(out["det_logit"].float()).cpu())
        if mc:
            enable_mc_dropout(model); ps = []
            for _ in range(mc):
                with ac: ps.append(torch.sigmoid(fwd(model, b)["det_logit"].float()))
            ps = torch.stack(ps); R["mc_mean"].append(ps.mean(0).cpu()); R["mc_std"].append(ps.std(0).cpu())
            pm = ps.mean(0).clamp(1e-6, 1 - 1e-6)
            R["entropy"].append((-(pm * pm.log() + (1 - pm) * (1 - pm).log())).cpu())
            model.eval()
        for k in ("y", "y_mask", "slice_lab", "slice_valid", "burden_t"): R[k].append(b[k].float().cpu())
        if "sev_logits" in out: R["sev_prob"].append(torch.sigmoid(out["sev_logits"].float()).cpu())
        for k in ("slice_prob", "burden_ext", "burden_vol", "burden_head"):
            if k in out: R[k].append(out[k].float().cpu())
        if keep_heat and "heat" in out: R["heat"].append(out["heat"].float().cpu().half())
    return {k: (v if k == "sid" else torch.cat(v).numpy()) for k, v in R.items()}

def det_metrics(R):
    y, p = R["y"][:, 0], R["det_prob"]
    return dict(auc=safe_auc(y, p), ap=float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else float("nan"),
                acc=float(((p >= cfg.thr_det) == (y > 0.5)).mean()))

def train_model(model, tr_ids, va_ids, cfg, epochs=None, tag="model", fwd=None, log=print, max_batches=None):
    """Generic trainer: works for PENet and for baseline models (loss skips missing heads)."""
    epochs = epochs or cfg.epochs; fwd = fwd or (lambda m, b: m(b)); model.to(DEVICE)
    dl_tr, dl_va = make_loader(tr_ids, cfg, True), make_loader(va_ids, cfg, False)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    steps = max(1, epochs * (len(dl_tr) if not max_batches else min(len(dl_tr), max_batches)) // cfg.accum)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=steps + 5, pct_start=0.15)
    use_amp = DEVICE.type == "cuda"; scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best, bad, hist = -1.0, 0, []; path = f"{cfg.ckpt_dir}/{tag}.pt"
    for ep in range(epochs):
        model.train(); t0 = time.time(); agg = defaultdict(float); n = 0
        for i, b in enumerate(dl_tr):
            if max_batches and i >= max_batches: break
            b = to_dev(b)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                out = fwd(model, b); loss, parts = compute_loss(out, b, cfg)
            scaler.scale(loss / cfg.accum).backward()
            if (i + 1) % cfg.accum == 0 or i + 1 == len(dl_tr) or (max_batches and i + 1 >= max_batches):
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if sched.last_epoch < steps + 3: sched.step()
            for k, v in parts.items(): agg[k] += v
            n += 1
        R = predict(model, dl_va, fwd=fwd); dm = det_metrics(R)
        sf1 = slice_f1(R["slice_prob"], R["slice_lab"], R["slice_valid"], cfg.thr_slice) if "slice_prob" in R else float("nan")
        rec = dict(epoch=ep, **{f"loss_{k}": v / max(n, 1) for k, v in agg.items()}, val_auc=dm["auc"], val_ap=dm["ap"], val_slice_f1=sf1, sec=time.time() - t0)
        hist.append(rec); log(" | ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in rec.items()))
        score = dm["auc"] if not math.isnan(dm["auc"]) else -rec["loss_det"]
        if score > best: best, bad = score, 0; torch.save(model.state_dict(), path)
        else:
            bad += 1
            if bad >= cfg.patience: log("early stop"); break
    model.load_state_dict(torch.load(path, map_location=DEVICE)); return model, hist, best